# Transfer learning with torchvision.models

_PyTorch_

**Start from a model pretrained on millions of images, then teach it your task with very little data.**

A deep image network learns its features in layers: early layers detect edges and colors, middle layers detect textures and parts, late layers detect whole objects. The early and middle features are generic — useful for almost any image task. Transfer learning keeps those learned layers (the backbone) and only replaces the last layer (the head) that maps features to your specific classes.

---

This notebook walks through transfer learning one step at a time. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **face images, not table columns** — each sample is a 64x64 grayscale picture.

In [ ]:
from sklearn.datasets import fetch_olivetti_faces

faces = fetch_olivetti_faces()
print("image array:", faces.images.shape, " labels:", faces.target.shape)
samples = list(zip(faces.images, faces.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

We take a ResNet-18 pretrained on ImageNet and adapt it to a 5-class task. We build it in four steps: (1) load the pretrained model, (2) freeze the backbone and swap the head, (3) train just the head, (4) optionally unfreeze and fine-tune the whole network.

### Step 1 — Load a pretrained model and its preprocessing

`ResNet18_Weights.DEFAULT` downloads both the architecture and the best available ImageNet weights — the result of training on millions of labeled images. The weights object also carries the *exact* preprocessing the model was trained with (resize, center-crop, ImageNet normalization); feeding it differently-scaled inputs would collapse accuracy.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet18, ResNet18_Weights

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

NUM_CLASSES = 5   # <- your number of classes

# ---- 1. LOAD A PRETRAINED MODEL ----
weights = ResNet18_Weights.DEFAULT          # best available ImageNet weights
model = resnet18(weights=weights)           # downloads architecture + trained weights

# The weights object carries the EXACT preprocessing the model expects.
# Apply this to every input image (resize, center-crop, ImageNet normalization).
preprocess = weights.transforms()
print("expected transforms:\n", preprocess)

### Step 2 — Freeze the backbone and swap in a new head

Setting `requires_grad = False` on every parameter tells autograd not to update the pretrained layers — we keep their generic edge/texture/part features intact. Then we replace the final `fc` layer with a fresh `nn.Linear` sized to our 5 classes; a new module trains by default, so it is the only thing that learns. We hand the optimizer **only** the head's parameters, not the whole network.

In [ ]:
# ---- 2. FREEZE THE BACKBONE (feature-extraction mode) ----
for param in model.parameters():
    param.requires_grad = False             # autograd will not update these

# ---- 3. SWAP THE FINAL LAYER FOR A NEW HEAD ----
in_features = model.fc.in_features          # 512 for resnet18
model.fc = nn.Linear(in_features, NUM_CLASSES)   # fresh layer -> requires_grad=True
model = model.to(device)

trainable = [n for n, p in model.named_parameters() if p.requires_grad]
print("trainable parameters:", trainable)   # only 'fc.weight', 'fc.bias'

# ---- 4. OPTIMIZER SEES ONLY THE HEAD ----
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)   # NOT model.parameters()
criterion = nn.CrossEntropyLoss()           # expects raw logits + class indices

### Step 3 — Train just the head

We put the whole model in `eval()` mode so the frozen backbone's batch-norm uses its stored running stats instead of recomputing them from tiny batches — then flip just the head back to `train()`. The training step is the usual cycle (zero grads, forward, loss, backward, step), but gradients now flow only into the head. Real projects swap the fake tensors for `datasets.ImageFolder(root, transform=preprocess)` + a `DataLoader`.

In [ ]:
# ---- 5. A SHORT TRAINING LOOP ----
# Tiny synthetic 'dataset' so this runs anywhere. resnet18 wants 3x224x224 inputs.
# In a real project: datasets.ImageFolder(root, transform=preprocess) + DataLoader.
x = torch.randn(16, 3, 224, 224, device=device)         # 16 fake images
y = torch.randint(0, NUM_CLASSES, (16,), device=device) # 16 fake labels

model.eval()        # backbone batchnorm uses stored running stats (frozen)
model.fc.train()    # but the head is training
for epoch in range(5):
    optimizer.zero_grad()           # clear last step's grads (they accumulate!)
    logits = model(x)               # forward pass
    loss = criterion(logits, y)     # cross-entropy on logits (no softmax needed)
    loss.backward()                 # grads flow only into the head
    optimizer.step()                # update the head
    print(f"epoch {epoch}: loss = {loss.item():.4f}")

### Step 4 — Optionally fine-tune the whole network

Once the head is roughly trained, you can unfreeze the backbone and train everything together — but with a **much smaller** learning rate (`1e-4`). The pretrained weights are already good, so we only want to nudge them gently; a large rate here would overwrite the valuable ImageNet features and hurt accuracy.

In [ ]:
# ---- 6. SWITCH TO FULL FINE-TUNING (optional phase two) ----
# Unfreeze the backbone and train everything with a SMALL learning rate so the
# good pretrained weights only adjust gently.
for param in model.parameters():
    param.requires_grad = True
optimizer = optim.Adam(model.parameters(), lr=1e-4)   # small lr for fine-tuning
model.train()
optimizer.zero_grad()
loss = criterion(model(x), y)
loss.backward()
optimizer.step()
print("after one fine-tune step:", loss.item())

## Visualize the data & results

_Transfer learning should win most when labels are scarce. Using frozen learned features (a stand-in for a pretrained backbone) vs raw pixels, how does test accuracy on a real face dataset grow as we give the classifier more labeled images?_

### Step 1 — Load real faces and build a frozen feature extractor

We use 400 real face photos of 40 people. To mimic a frozen pretrained backbone without downloading one, we learn an eigenface PCA basis **once** on the pool and treat its 100 whitened components as fixed features. A held-out test set is split off up front so every label-budget experiment is scored on the same unseen faces.

In [ ]:
import numpy as np
from sklearn.datasets import fetch_olivetti_faces
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Real images: 400 face photos of 40 people, 64x64 = 4096 raw pixels each.
faces = fetch_olivetti_faces()
X, y = faces.data, faces.target

# Hold out a fixed test set; sample tiny label budgets from the rest.
X_pool, X_te, y_pool, y_te = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=0)

# TRANSFER (feature-extraction mode): a FROZEN backbone. An eigenface PCA basis
# learned once on the face manifold stands in for a pretrained backbone's frozen
# features; we train only a linear head on the few labels.
center = StandardScaler(with_std=False).fit(X_pool)
backbone = PCA(n_components=100, whiten=True, random_state=0).fit(center.transform(X_pool))
def features(Z):
    return backbone.transform(center.transform(Z))

### Step 2 — Define the label-budget experiment

`sample` draws a fixed number of labeled images per person; `accuracy` trains a linear classifier and scores it on the held-out test set, averaging over 20 random draws to smooth out luck. The only difference between the two modes is the input: **transfer** feeds the frozen PCA features, **from scratch** feeds raw pixels with no learned features.

In [ ]:
classes = np.unique(y_pool)
def sample(per_class, seed):
    rng = np.random.RandomState(seed); idx = []
    for c in classes:
        ci = np.where(y_pool == c)[0]
        idx.extend(rng.choice(ci, size=min(per_class, len(ci)), replace=False))
    return np.array(idx)

def accuracy(per_class, mode):
    accs = []
    for seed in range(20):                       # average over 20 random samples
        idx = sample(per_class, seed)
        Xs, ys = X_pool[idx], y_pool[idx]
        if mode == "transfer":                   # frozen features + linear head
            Ftr, Fte = features(Xs), features(X_te)
        else:                                    # FROM SCRATCH: raw pixels, no learned features
            Ftr, Fte = Xs, X_te
        clf = LogisticRegression(max_iter=3000, C=1e4).fit(Ftr, ys)
        accs.append(clf.score(Fte, y_te))
    return round(float(np.mean(accs)), 3)

### Step 3 — Sweep the label budget and compare

We sweep from 1 to 7 labeled images per person and print accuracy for both modes. The gap is the point of transfer learning: with frozen pretrained features the classifier reaches high accuracy from a handful of labels, while training from raw pixels lags behind — and the advantage is largest when labels are scarcest.

In [ ]:
budgets = [1, 2, 3, 4, 5, 7]                      # labeled images PER PERSON
total = [b * 40 for b in budgets]                # 40 people
print("labeled images total:", total)
print("transfer    :", [accuracy(b, "transfer") for b in budgets])
print("from scratch :", [accuracy(b, "scratch") for b in budgets])
# labeled images total: [40, 80, 120, 160, 200, 280]
# transfer    : [0.548, 0.707, 0.815, 0.858, 0.893, 0.925]
# from scratch : [0.506, 0.655, 0.727, 0.761, 0.778, 0.85]

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Type this in Colab. Load resnet18 with ResNet18_Weights.DEFAULT and print model.fc. Then print model.fc.in_features and model.fc.out_features so you see the ImageNet head's exact shape before you replace it.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Build with resnet18(weights=ResNet18_Weights.DEFAULT). — _DEFAULT downloads the architecture plus the best ImageNet weights._
- Read model.fc.in_features and .out_features. — _You need the input size (512) to size a new head, and the 1000 outputs are ImageNet's classes._

**Answer:** import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights
model = resnet18(weights=ResNet18_Weights.DEFAULT)
print(model.fc)                  # Linear(in_features=512, out_features=1000, bias=True)
print(model.fc.in_features)      # 512
print(model.fc.out_features)     # 1000

</details>

**Problem 2.** Type this in Colab. Freezing pitfall. On the loaded resnet18, count how many parameter tensors have requires_grad=True before freezing. Then freeze every parameter with requires_grad=False and count again. Predict the second count before running.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Count sum(p.requires_grad for p in model.parameters()) first. — _Out of the box every tensor is trainable — that is "training from scratch"._
- Loop for p in model.parameters(): p.requires_grad = False, then recount. — _Freezing makes autograd skip those tensors; the count drops to 0._

**Answer:** before = sum(p.requires_grad for p in model.parameters())
print(before)    # 62  (all tensors trainable)
for p in model.parameters():
    p.requires_grad = False
after = sum(p.requires_grad for p in model.parameters())
print(after)     # 0  (backbone frozen)

</details>

**Problem 3.** Type this in Colab. After freezing the backbone, replace the head with nn.Linear(in_features, 5) for a 5-class task. Then print the names of the parameters that still require grad — confirm only the new fc trains.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Read in_features = model.fc.in_features, then assign a fresh model.fc = nn.Linear(in_features, 5). — _A brand-new nn.Linear is created with requires_grad=True by default._
- List [n for n, p in model.named_parameters() if p.requires_grad]. — _Only the new head should appear — the frozen backbone is excluded._

**Answer:** in_features = model.fc.in_features          # 512
model.fc = nn.Linear(in_features, 5)        # fresh head -> requires_grad=True
trainable = [n for n, p in model.named_parameters() if p.requires_grad]
print(trainable)   # ['fc.weight', 'fc.bias']  -- only the head trains

</details>

**Problem 4.** Type this in Colab. Optimizer pitfall. Create the optimizer for feature-extraction mode so it touches only the head, then print how many parameter tensors it manages. (Hint: pass model.fc.parameters(), not model.parameters().)

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Build optim.Adam(model.fc.parameters(), lr=1e-3). — _Handing it the whole network would waste bookkeeping on frozen, gradient-less params._
- Count len(optimizer.param_groups[0]['params']). — _It should be 2 — the head's weight and bias only._

**Answer:** import torch.optim as optim
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)   # NOT model.parameters()
print(len(optimizer.param_groups[0]['params']))   # 2  (fc.weight, fc.bias)

</details>

**Problem 5.** Type this in Colab. Use the model's expected transforms. Grab preprocess = ResNet18_Weights.DEFAULT.transforms() and print it. Then apply it to a fake PIL-sized tensor input by building a random (3, 300, 300) uint8 image and confirming the transform resizes it to the 3&times;224&times;224 the weights expect.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Call weights.transforms() to get the exact resize/crop/normalize pipeline. — _The weights were trained on this specific preprocessing; mismatched input collapses accuracy._
- Run it on a sample image and print the output shape. — _It center-crops to 224 and normalizes with ImageNet statistics._

**Answer:** import torch
from torchvision.models import ResNet18_Weights
weights = ResNet18_Weights.DEFAULT
preprocess = weights.transforms()
print(preprocess)   # ImageClassification(crop_size=[224], resize_size=[256], mean=..., std=...)
img = torch.randint(0, 256, (3, 300, 300), dtype=torch.uint8)
out = preprocess(img)
print(out.shape)    # torch.Size([3, 224, 224])
print(out.dtype)    # torch.float32  (normalized)

</details>

**Problem 6.** Type this in Colab. model.eval() for the frozen backbone. Put the frozen-backbone model in eval mode, then put just the head back in train mode. Print model.training and model.fc.training to confirm the split.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Call model.eval() on the whole model. — _So the backbone's batch-norm uses stored running stats instead of recomputing from tiny batches._
- Call model.fc.train() to re-enable training only on the head. — _The new head is what you actually optimize._

**Answer:** model.eval()        # backbone batchnorm -> stored running stats
model.fc.train()    # but the head trains
print(model.training)      # False
print(model.fc.training)   # True

</details>

**Problem 7.** Type this in Colab. One feature-extraction training step. With the frozen backbone + new 5-class head, run a single step on a fake batch x = torch.randn(8, 3, 224, 224), y = torch.randint(0, 5, (8,)) using CrossEntropyLoss and the head-only optimizer. Remember zero_grad(). Print the loss.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Forward through the model, score with CrossEntropyLoss on raw logits. — _Cross-entropy wants logits and integer class indices, no softmax._
- optimizer.zero_grad(), loss.backward(), optimizer.step(). — _Gradients flow only into the head; zeroing prevents them accumulating across steps._

**Answer:** import torch
torch.manual_seed(0)
criterion = nn.CrossEntropyLoss()
x = torch.randn(8, 3, 224, 224)
y = torch.randint(0, 5, (8,))
optimizer.zero_grad()
logits = model(x)               # (8, 5) raw logits
loss = criterion(logits, y)
loss.backward()                 # grads only in fc
optimizer.step()
print(round(loss.item(), 4))    # ~1.6  (near ln(5), random head start)

</details>

**Problem 8.** Type this in Colab. Switch to fine-tuning. Unfreeze the whole backbone, rebuild the optimizer over model.parameters() with a small learning rate 1e-4, switch to model.train(), and run one step on the same fake batch. Print the loss. Note why 1e-2 would be the wrong rate here.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Set requires_grad=True on every param and rebuild optim.Adam(model.parameters(), lr=1e-4). — _Now the whole network adapts, but gently — pretrained weights are already good._
- Call model.train() and run one full step. — _A large rate like 1e-2 would take big steps and wreck the pretrained features._

**Answer:** for p in model.parameters():
    p.requires_grad = True
optimizer = optim.Adam(model.parameters(), lr=1e-4)   # small lr for fine-tuning
model.train()
optimizer.zero_grad()
loss = criterion(model(x), y)
loss.backward()
optimizer.step()
print(round(loss.item(), 4))   # another small loss; backbone now adapting gently
# lr=1e-2 here would overwrite the good ImageNet features and accuracy would drop.

</details>